# Problem Set 8

[PSet 8](pset8.pdf)

In [1]:
import sympy as sp
import sympy.physics.mechanics as spm
import sympy.physics.vector as spv
from sympy.physics.vector.printing import init_vprinting
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


import IPython

# Import IPython display for proper LaTeX formatting
from IPython.display import display, Math, Markdown

# Initialize symbols
sp.init_printing()

# Enable dot notation printing for dynamicsymbols
init_vprinting(use_latex="mathjax")

HALF = sp.S.Half

In [2]:
def reference_frame(frame: str, x=r"\imath", y=r"\jmath", z=r"k") -> spm.ReferenceFrame:
    """Create a SymPy reference frame with custom basis vector labels.

    Parameters
    ----------
    frame : str
        The name of the reference frame.
    x, y, z : str
        Labels for the basis vectors.
    """
    return spm.ReferenceFrame(
        frame,
        latexs=(
            rf"\;{{}}^\mathcal{{{frame}}}\hat{{{x}}}",
            rf"\;{{}}^\mathcal{{{frame}}}\hat{{{y}}}",
            rf"\;{{}}^\mathcal{{{frame}}}\hat{{{z}}}",
        ),
    )


def reference_frame_circular(name: str, angle=r"theta") -> spm.ReferenceFrame:
    """Create a circular reference frame with radial and angular basis labels.

    Parameters
    ----------
    name : str
        Name of the new reference frame.
    angle : str, optional
        Symbol or label used for the angular basis vector, by default "theta".
    """
    return reference_frame(name, x=r"r", y=rf"\{angle}", z=r"e_z")

## Problem 8.1 

Spring-Loop-the-Loop
A small block of mass m is pushed against a spring with spring constant k and held in
place with a catch. The spring compresses an unknown distance x. When the catch
is removed, the block leaves the spring and slides along a frictionless circular loop of
radius R


![Slide Down an Inclined Plane](../figures/PS0801-spring-block-loop.jpg)

When the block reaches the top of the loop, the force of the loop on the block (the
normal force) is equal to twice the gravitational force on the mass. How far was the
spring initially compressed? Write your answer using some or all of the following: g,
k, R, and m.

In [3]:
(
    m,  # mass of the object
    ell_0,  # length of the inclined plane
    k, # spring constant
    x, # compression of the spring
    g,  # acceleration due to gravity
    R  # distance it takes the object to stop measured from the bottom of the incline
) = sp.symbols("m ell_0 k x g R", real=True, positive=True)

theta = spm.dynamicsymbols("theta")

In [4]:
N = reference_frame("N")
M = reference_frame("M", x=r"r", y=r"\theta", z=r"e_z")
M.orient_axis(N, N.z, theta)   

vec_R = R * M.x
vec_v = vec_R.dt(N)
vec_a = vec_v.dt(N)

display(Math(rf"\text{{Position vector: }} {spv.vlatex(vec_R)}"))
display(Math(rf"\text{{Velocity vector: }} {spv.vlatex(vec_v)}"))
display(Math(rf"\text{{Acceleration vector: }} {spv.vlatex(vec_a)}")) 

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [5]:
# Define CME for top of loop
vf = sp.symbols("v_f", real=True, positive=True) # Velocity at the top of the loop

# Conservation of mechanical energy(CME) for spring at moment of release
CMEeqn = sp.Eq(sp.S.Half * m * vf**2 + m * g * 2 * R, sp.S.Half * k * x**2 )
vfsq_sol = sp.solve(CMEeqn, vf**2)[0]
display(Math(rf"\text{{Speed squared at the top of the loop: }} {spv.vlatex(vfsq_sol)}"))

<IPython.core.display.Math object>

In [6]:
# Define the forces acting on the object in circular loop
F_gravity = m * g * N.x
Normal_mag = 2 * m * g  # Normal force magnitude
F_Normal = Normal_mag * (-M.x)
F_total = F_gravity + F_Normal

# WET for the object at an angle pi in the circular loop.
# Top of the loop is theta=pi, bottom of the loop is theta=0

# N2L
N2Leqn = spm.msubs(
    sp.Eq(F_total.to_matrix(N), (m * vec_a.to_matrix(N))), {theta: sp.pi}
)
N2Leqn_solved = sp.solve(N2Leqn, [theta.diff() ** 2, theta.diff().diff()], dict=True)[0]

# Velocity squared at the top of the loop
x_solution = sp.solve(
    sp.Eq(R**2 * N2Leqn_solved[theta.diff() ** 2], vfsq_sol), x**2, dict=True
)[0]
x_solution_sqrt = sp.sqrt(x_solution[x**2].simplify(), evaluate=False)
display(
    Math(rf"\text{{Spring compression }} x: \boxed{{{spv.vlatex(x_solution_sqrt)}}}")
)

<IPython.core.display.Math object>

## Problem 8.2 Sling Shot

A ball of negligible size and mass m hangs from a string of length l. 
It is hit in such a way that it then travels in a vertical circle. 
The initial speed of the ball after being struck is v0. The goal of the 
first part of this problem is to find the tension in the string when the 
ball is at the top of the circle. You may assume that there are no
external forces other than gravity doing work on the ball and string. 
Let g denote the magnitude of the gravitational constant.

![Sling Shot](../figures/PS0802-sling-shot.jpg)

(a) Find the tension in the string when the ball is at the top of the circle. 
Express your answer in terms of some or all of the following: $m, g, \ell,  v_0$.


(b) When the ball is exactly at the top of the circle, it detaches from 
the string and follows the trajectory shown in the figure above. When the 
ball returns to the level of the bottom of the circle, it is a distance 
d from the bottom of the circle. Find the distance d. 

Express your answer in terms of $m, g, \ell,  v_0$ as needed.

### Kinetics

![Sling shot kinetics](../figures/PS0802-sling-shotFBD.jpg)

In [7]:
(
    m,  # mass of the object
    g,  # acceleration due to gravity
    ell, # length of string
    v0 # initial velocity of the object
) = sp.symbols("m g ell v0", real=True, positive=True)

theta = spm.dynamicsymbols("theta")

In [8]:
N = reference_frame("N")
M = reference_frame("M", x=r"r", y=r"\theta", z=r"e_z")
M.orient_axis(N, N.z, theta)   

vec_R = ell * M.x
vec_v = vec_R.dt(N)
vec_a = vec_v.dt(N)

display(Math(rf"\text{{Position vector: }} {spv.vlatex(vec_R)}"))
display(Math(rf"\text{{Velocity vector: }} {spv.vlatex(vec_v)}"))
display(Math(rf"\text{{Acceleration vector: }} {spv.vlatex(vec_a)}")) 

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [9]:
# a.
# CME for ball from bottom of loop to top of loop
vtop = sp.symbols("v_top", real=True, positive=True)  # Velocity at the top of the loop
CMEball = sp.Eq(HALF * m * v0**2, HALF * m * vtop**2 + m * g * 2 * ell)
vtop_sq_sol = sp.solve(CMEball, vtop**2)[0]
display(
    Math(rf"\text{{Speed squared at the top of the loop: }} {spv.vlatex(vtop_sq_sol)}")
)

theta_sq = sp.solve(
    sp.Eq(vec_v.magnitude() ** 2, vtop_sq_sol), theta.diff() ** 2, dict=True
)[0]

display(
    Math(
        rf"\text{{Speed squared at the top of the loop: }} {spv.vlatex(theta_sq[theta.diff()**2])}"
    )
)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [10]:
# N2L for the ball at the top of the loop
Tension = sp.symbols(
    "T", real=True, positive=True
)  # Tension in the string at the top of the loop
F_gravity = m * g * N.x
F_total = F_gravity + Tension * (-M.x)

N2Leqn = spm.msubs(
    sp.Eq(F_total.to_matrix(N), (m * vec_a).to_matrix(N)), {theta: sp.pi}
)
Tension_solution = sp.solve(N2Leqn, Tension, dict=True)[0][Tension]
Tension_sol = spm.msubs(Tension_solution, theta_sq).simplify()
display(
    Math(
        rf"\text{{Tension in the string at the top of the loop: }} \boxed{{{spv.vlatex(Tension_sol)}}}"
    )
)

<IPython.core.display.Math object>

In [ ]:
# b. Calculate time to fall from the top of the loop to the bottom of the loop.

t = sp.symbols("t", real=True, positive=True)  # Time to fall from top to bottom of loop
fall_eqn = sp.Eq(2 * ell, HALF * g * t**2)
time_sq_to_fall = sp.solve(fall_eqn, t**2)[0]
d = sp.sqrt(time_sq_to_fall * vtop_sq_sol, evaluate=False)

display(
    Math(
        rf"\text{{Horizontal distance traveled by the ball from top to bottom of the loop: }}"
        rf"\boxed{{{spv.vlatex(d)}}}"
    )
)

<IPython.core.display.Math object>